# Limpieza de datos

En este notebook limpiamos las tablas de usuarios, características de audio y escuchas. Usamos los criterios del notebook 01_exploracion_calidad_ml.ipynb. Después guardamos las tablas en Parquet.

## 1. Preparación del entorno

Creamos una vista para cada archivo original. Las rutas y la configuración de DuckDB están en `config.py`

In [1]:
from config import conectar, ruta_sql

con = conectar()

# Vistas sobre los archivos originales
con.execute(f"CREATE OR REPLACE VIEW events AS SELECT * FROM {ruta_sql('events')}")
con.execute(f"CREATE OR REPLACE VIEW users AS SELECT * FROM {ruta_sql('users')}")
con.execute(f"CREATE OR REPLACE VIEW acoustic_features AS SELECT * FROM {ruta_sql('acoustic_features')}")

# Ejecuta una consulta SQL y devuelve el resultado como DataFrame.
def q(consulta):
    return con.sql(consulta).df()

q("SELECT 'todo listo' AS estado")

,estado
0,todo listo


## 2. Limpieza de las tablas

No eliminamos usuarios ni canciones por los valores fuera de rango: los reemplazamos por nulos. En la tabla de escuchas sacamos la de 1970 y las de los días en que un usuario superó las 720 escuchas.

Usamos `COPY ... TO` para guardar directamente el resultado de cada consulta en un archivo Parquet.

### 2.1 Usuarios

Consideramos válidas las edades entre 10 y 90 años. Las que quedan afuera de ese rango, incluido el -1, las pasamos a nulo. En género conservamos `m` y `f`, y el resto queda como "no informado". También unificamos UK y GB usando GB para el Reino Unido.

Para hacer estos cambios usamos `CASE WHEN`, que funciona de manera parecida a un `if`: si se cumple la condición, devuelve el valor indicado después de `THEN`; si no, devuelve el de `ELSE`.

In [2]:
con.execute(f"""
    COPY (
        SELECT user_id,
               CASE WHEN country = 'UK' THEN 'GB' ELSE country END        AS country,
               CASE WHEN age BETWEEN 10 AND 90 THEN age ELSE NULL END       AS age,
               CASE WHEN gender IN ('m', 'f') THEN gender
                    ELSE 'no informado' END                                AS gender,
               playcount,
               registered_unixtime
        FROM users
    ) TO {ruta_sql('users_clean')} (FORMAT parquet)
""")

# Revisamos cómo quedaron las columnas que modificamos
q(f"""
    SELECT COUNT(*)                                    AS usuarios,
           COUNT(*) FILTER (WHERE age IS NULL)          AS edad_nula,
           COUNT(*) FILTER (WHERE gender = 'no informado') AS genero_no_informado,
           COUNT(*) FILTER (WHERE country = 'UK')       AS quedan_uk
    FROM {ruta_sql('users_clean')}
""")

,usuarios,edad_nula,genero_no_informado,quedan_uk
0,55190,12481,4747,0


Quedaron los 55.190 usuarios. De ellos, 12.481 tienen la edad en nulo y 4.747 tienen el género como "no informado". Ya no quedan casos con el código UK.

### 2.2 Características de audio

Pasamos a nulo los valores de `tempo` iguales a 0 y los de `loudness` iguales a -60 dB. No eliminamos las canciones.

Dejamos afuera `key` (nota musical) y `mode` (modo mayor o menor) porque no las vamos a utilizar en los modelos. Son variables categóricas, aunque estén guardadas como números.

Con `NULLIF` podemos reemplazar un valor puntual por un nulo. Por ejemplo, `NULLIF(tempo, 0)` devuelve nulo si el tempo es 0 y, si no, conserva el valor original.

In [3]:
con.execute(f"""
    COPY (
        SELECT track_id,
               danceability,
               energy,
               NULLIF(loudness, -60) AS loudness,
               speechiness,
               acousticness,
               instrumentalness,
               liveness,
               valence,
               NULLIF(tempo, 0)      AS tempo
        FROM acoustic_features
    ) TO {ruta_sql('acoustic_clean')} (FORMAT parquet)
""")

# COUNT(columna) no cuenta los nulos
q(f"""
    SELECT COUNT(*)                  AS canciones,
           COUNT(*) - COUNT(tempo)    AS tempo_nulo,
           COUNT(*) - COUNT(loudness) AS loudness_nulo
    FROM {ruta_sql('acoustic_clean')}
""")

,canciones,tempo_nulo,loudness_nulo
0,3471884,2651,209


### 2.3 Escuchas

eliminamos las escuchas anteriores a 2005 porque quedan fuera del período que analizamos. También contamos cuántas escuchas registró cada usuario por día. Si superó las 720, sacamos las escuchas de ese día y conservamos las de sus otros días. Guardamos el resultado en events_clean.parquet.

In [4]:
con.execute(f"""
    COPY (
        -- Escuchas por usuario y día
        WITH dias AS (
            SELECT user_id, "timestamp" // 86400 AS dia, COUNT(*) AS escuchas_dia
            FROM events
            GROUP BY user_id, dia
        ),
        -- Días que superan el límite
        dias_anomalos AS (
            SELECT user_id, dia
            FROM dias
            WHERE escuchas_dia > 720
        )
        -- Conservamos las escuchas que pasan los filtros
        SELECT e.user_id, e.artist_id, e.album_id, e.track_id, e."timestamp"
        FROM events e
        LEFT JOIN dias_anomalos a
               ON e.user_id = a.user_id
              AND e."timestamp" // 86400 = a.dia
        WHERE a.user_id IS NULL
          AND e."timestamp" >= 1104537600
    ) TO {ruta_sql('events_clean')} (FORMAT parquet, COMPRESSION zstd)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### 2.4 Control de la limpieza

Comparamos la cantidad de escuchas antes y después de la limpieza para saber qué porcentaje de los datos eliminamos.

In [6]:
antes   = con.execute("SELECT COUNT(*) FROM events").fetchone()[0]
despues = con.execute(f"SELECT COUNT(*) FROM {ruta_sql('events_clean')}").fetchone()[0] #fetchone()[0] obtiene el primer valor que devuelve la consulta (en este caso, el total de filas).


print(f"Escuchas originales: {antes:>15,}")
print(f"Escuchas limpias:    {despues:>15,}")
print(f"Eliminadas:          {antes - despues:>15,}  ({(antes - despues) / antes * 100:.2f}%)")

Escuchas originales:     351,469,333
Escuchas limpias:        350,741,953
Eliminadas:                  727,380  (0.21%)


Se eliminaron 727.380 escuchas, que representan el 0,21% del total.

Los archivos que quedan guardados en `data/clean/` son:\
 `users_clean.parquet`\
 `acoustic_clean.parquet`\
 `events_clean.parquet` 